# Classify the ICESat-2 Photons with C-SHELPh (optional)

*Section 10 — Case study: ICESat-2 bathymetry, notebook 5 of 5.* Previous: `04_landsat_scenes_bathymetry`. Next: `11_case_study_species_distribution/01_prepare_datasets`.

**This notebook is optional.** The earlier notebooks in this section use ICESat-2 depth points which have already been extracted for you (`../data/is2_bathy/is2_bermuda_depth.gpkg`). This notebook shows how such depths are derived from the raw ICESat-2 ATL03 photon data.

> **Requirements:** this notebook needs the external [C-SHELPh](https://github.com/nmt28/C-SHELPh) package (`cshelph`), which is not part of the tutorial environment (install it with `pip install cshelph`), and ICESat-2 ATL03 HDF5 files downloaded with `12_downloading_data/08_icepyx_download_icesat2` (which needs a NASA Earthdata login). Run that notebook first: it saves the files to `12_downloading_data/outputs/icesat2_dwnld/<roi>/<year>/*.h5`. The cells that need `cshelph` or the downloaded data are tagged `skip-execution` so the automated test run skips them.

**You will learn:**
- how to iterate through a directory tree of downloaded ICESat-2 ATL03 granules;
- how to run the C-SHELPh photon classifier for each laser (beam pair) to extract bathymetric photons;
- how to record which granules/lasers were processed successfully.

**Before you start:** notebook `01_visualise_icesat2` of this section, and the download notebook `12_downloading_data/08_icepyx_download_icesat2`.

**Data:** the ATL03 `*.h5` files in `../12_downloading_data/outputs/icesat2_dwnld/` (not supplied in `data/`).

**Run time:** not run as part of the tests (skipped cells); with the data and `cshelph` installed it depends on the number of granules (several minutes per granule).

The next stage is to classify the ICESat-2 photons to identify the bathymetry points from the ATL03 data. To achieve this we'll use the C-SHELPh module (https://github.com/nmt28/C-SHELPh). This notebook uses no rsgispy functions; C-SHELPh writes its outputs (classified photons and depths) next to the input files.

## Import Modules

In [1]:
import json
import pathlib

In [ ]:
# Needs the external cshelph package (not installed in the tutorial environment).
import cshelph.run_cshelph

## Iterate through Data Directories

The download notebook stores the granules as `<roi>/<year>/*.h5` under its own `outputs/icesat2_dwnld` folder.

In [2]:
base_data_dir = "../12_downloading_data/outputs/icesat2_dwnld"

In [3]:
base_path = pathlib.Path(base_data_dir)

In [ ]:
roi_paths = [f for f in base_path.iterdir() if f.is_dir()]

In [ ]:
for roi_path in roi_paths:
    year_paths = [f for f in roi_path.iterdir() if f.is_dir()]
    out_info = dict()
    for year_path in year_paths:
        h5_paths = year_path.glob("*.h5")
        for h5_path in h5_paths:
            print(h5_path)
            h5_filename = h5_path.stem
            print(f"\t{h5_filename}")
            out_info[h5_filename] = dict()
            for laser in [1, 2, 3]:
                try:
                    cshelph.run_cshelph.run_cshelph(
                        input_h5_file=str(h5_path),
                        laser_num=laser,
                        thresh=20,
                        threshlist=None,
                        output=None,
                        lat_res=10,
                        h_res=0.5,
                        water_temp=25,
                        start_lat=None,
                        end_lat=None,
                        min_buffer=-40,
                        max_buffer=5,
                        surface_buffer=-0.5,
                    )
                    out_info[h5_filename][laser] = "OK"
                except Exception as e:
                    out_info[h5_filename][laser] = str(e)
    out_info_file = roi_path / "cshelph_processing.json"
    with open(out_info_file, "w") as fp:
        json.dump(
            out_info,
            fp,
            sort_keys=True,
            indent=4,
            separators=(",", ": "),
            ensure_ascii=False,
        )